# Session 4: Scraping with Beautiful Soup

We will scrape https://www.shortform.com/best-books/genre/best-scrapbooking-books-of-all-time and turn one webpage into a Pandas dataframe.

By the end, you should be able to:

- request a webpage
- check whether the request worked
- parse HTML
- identify a repeated container
- extract titles, prices, ratings, availability and links
- create and inspect a dataframe
- clean and save the scraped data

> Workflow: **Request → Parse → Select → Extract → Store → Validate**

## Before scraping

Understand the site's html structure.

## 1. Install packages if needed

### This cell installs the three external libraries used in the notebook:

- `requests` downloads the webpage.
- `beautifulsoup4` reads and searches the HTML.
- `pandas` turns the extracted information into a table.

In [5]:
# !python3 -m pip install requests beautifulsoup4 pandas

## 2. Import libraries

### We are loading four tools:

- `requests` will fetch the webpage.
- `BeautifulSoup` will turn raw HTML into a searchable structure.
- `pandas`, shortened to `pd`, will create and analyse our dataframe.
- `urljoin` will turn incomplete links from the webpage into full URLs.

Running an import makes these tools available to the notebook.

In [6]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin

#### import requests
A Python library that sends requests to websites and downloads their content. Without it, Python has no way to ask a website for its contents.

#### from urllib.parse import urljoin
A function that combines a website's base URL with an incomplete (relative) link to create a full webpage address. We use it because many websites store links as relative paths instead of complete URLs.


Website
   │
   ▼
requests
(download page)
   │
   ▼
Beautiful Soup
(read HTML)
   │
   ▼
urljoin
(fix incomplete links)
   │
   ▼
Pandas
(store everything as a dataframe)

## 3. Store the URL

### We are saving the webpage address inside a variable called `url`.

The URL is text, so it must be placed inside quotation marks. Storing it in a variable means we can reuse the address later without typing it repeatedly.

The second line displays the value so we can confirm that the variable contains the expected address.

In [7]:
url = "https://www.shortform.com/best-books/genre/best-scrapbooking-books-of-all-time"
url

'https://www.shortform.com/best-books/genre/best-scrapbooking-books-of-all-time'

## 4. Request the webpage

This is the first time Python communicates with a website.

When we write:

```python
response = requests.get(url)
```

the following happens:

1. Python sends an HTTP **GET request** to the website.
2. The website's server receives that request.
3. The server sends back a **response**.
4. We store that response in a variable called `response`.

Think of it like ordering food:

- **You** → make the request.
- **Restaurant** → prepares the order.
- **Delivery bag** → the response.

The response contains much more than the webpage itself. It also contains:
- the status code
- headers
- cookies
- and the HTML that generated the page.

We haven't downloaded a spreadsheet—we've downloaded the webpage itself.


### What we are about to do

This cell sends an HTTP request to the website.

- `requests.get(url, timeout=30)` asks the server for the page stored in `url`.
- `timeout=30` prevents Python from waiting forever if the website does not respond.
- The server's reply is saved in a variable called `response`.
- Writing `response` on the final line displays a short summary, usually something such as `<Response [200]>`.

At this stage, `response` contains the status code, headers and webpage HTML.

In [8]:
response = requests.get(url, timeout=30)
response

<Response [200]>

### Optional: Check the response's status code.

A status code is the server's short message about what happened:

- `200` means the request succeeded.
- `403` means access was refused.
- `404` means the page was not found.
- `500` means the server encountered an error.

We should check this before trying to parse the page.

In [9]:
response.status_code

200

## 5. Look at the raw HTML

The browser turns HTML into something beautiful using CSS. But, Python doesn't see colours, buttons or layouts.
It only receives the underlying HTML.

The next two lines do two different jobs:

```python
html = response.text
```
stores the webpage's HTML inside a variable called `html`.

```python
print(html[:1000])
```

prints only the **first 1,000 characters**.

Why only the first thousand?

A webpage can contain tens of thousands of characters. Printing everything is a lot.

`[:1000]` is called **slicing** and simply means:

> "Show me the first thousand characters."

### Separate the webpage's HTML from the rest of the response.

- `response.text` contains the HTML as one long string of text.
- We store that string in a variable called `html`.
- `html[:1000]` uses slicing to keep only the first 1,000 characters.
- `print()` displays that shortened sample.

We inspect only the beginning because printing the full page would overwhelm the notebook.

In [10]:
html = response.text
print(html[:1000])

<!DOCTYPE html><html lang="en"><head><meta charset="utf-8">
<meta http-equiv="X-UA-Compatible" content="IE=edge">
<meta name="viewport" content="width=device-width,initial-scale=1,viewport-fit=cover">
<script data-prerender-keep="">(function(w,d,s,l,i){w[l]=w[l]||[];w[l].push({'gtm.start':
    new Date().getTime(),event:'gtm.js'});var f=d.getElementsByTagName(s)[0],
    j=d.createElement(s),dl=l!='dataLayer'?'&l='+l:'';j.async=true;j.src=
    'https://www.googletagmanager.com/gtm.js?id='+i+dl;f.parentNode.insertBefore(j,f);
    })(window,document,'script','dataLayer','GTM-TZKKGXF');</script>
<script async="" src="https://www.googletagmanager.com/gtag/js?id=UA-139186950-1" data-prerender-keep=""></script>
<script>window.dataLayer = window.dataLayer || [];
      function gtag(){dataLayer.push(arguments);}
      gtag('js', new Date());

      var gtagConfig = {
        // We track page views via Google Tag Manager UI configuration,
        // disable it here to avoid duplicate page views.

In [34]:
 print(html[10000:15000])

"><img src="https://images.amazon.com/images/P/1933516666.jpg"></a></div><div class="col-md-10 col-12 px-md-3 px-1"><div class="row"><div class="col-3 d-block d-md-none pr-0"><a href="https://www.amazon.com/gp/product/1933516666/?tag=allencheng-20" rel="nofollow" target="_blank"><img src="https://images.amazon.com/images/P/1933516666.jpg"></a></div><div class="col-9 col-md-12"><h2 class="display-4 mb-0 mt--2"><a href="https://www.amazon.com/gp/product/1933516666/?tag=allencheng-20" rel="nofollow" target="_blank">Life Artist</a></h2><p class="text-lg mb-0 subtitle"><a href="https://www.amazon.com/gp/product/1933516666/?tag=allencheng-20" rel="nofollow" target="_blank"></a></p><p class="justify-content-center byline pt-1"><span class="d-block d-md-inline">Creating Keepsakes</span><span class="d-none d-md-inline"> | </span><span class="sf-star-rating d-block d-md-inline"><span class="fas fa-star checked"></span><!----><!----><span class="fas fa-star checked"></span><!----><!----><span cla

Look for repeated structures such as:

```html
<article class="product_pod">
<p class="price_color">£51.77</p>
```

## 6. Parse the HTML

Right now `html` is just one very long string of text. Beautiful Soup converts that text into something we can search.

Think of it like this:

Before Beautiful Soup:
```
One giant wall of text
```

After Beautiful Soup:
```
A searchable tree of elements
```

Instead of searching through thousands of characters ourselves, we can now ask questions like:
- Find the first heading.
- Find every book.
- Find every price.
- Find every link.

### Turning the raw HTML string into a Beautiful Soup object.

- `html` is currently plain text.
- `"html.parser"` tells Beautiful Soup which parser to use.
- The parsed page is stored in `soup`.
- `type(soup)` checks what kind of Python object was created.

After this step, we can search the webpage by tags, classes and other HTML features.

In [35]:
soup = BeautifulSoup(html, "html.parser")
type(soup)

bs4.BeautifulSoup

## 7. Find the page heading

### What we are about to do

We are asking Beautiful Soup to find the first `<h1>` heading on the page.

- `soup.find("h1")` searches the parsed HTML.
- The matching HTML element is stored in `heading`.
- Displaying `heading` shows both the tag and its contents.

This is our first simple test that Beautiful Soup can locate an element successfully.

In [36]:
heading = soup.find_all("h1")

heading

[<h1 class="display-3 text-black font-weight-bold mb-3 text-md-left text-center" style="line-height: 1.2;">100 Best Scrapbooking Books of All Time </h1>]

### The `heading` variable currently contains an HTML element such as `<h1>All products</h1>`.

`get_text(strip=True)` removes the surrounding tags and returns only the visible words. `strip=True` also removes extra spaces and line breaks from the beginning and end.

In [37]:
heading_new = soup.find("h1")

heading_new

<h1 class="display-3 text-black font-weight-bold mb-3 text-md-left text-center" style="line-height: 1.2;">100 Best Scrapbooking Books of All Time </h1>

In [38]:
heading_new.get_text(strip=True)

'100 Best Scrapbooking Books of All Time'

In [39]:
print(heading_new.get_text(strip=True))

100 Best Scrapbooking Books of All Time


## 8. Find one book card

Every book lives inside:

```html
<article class="product_pod">
```

Every book has the **same HTML structure**. That repetition is exactly what makes scraping possible.
Rather than writing code twenty times, we teach Python how to recognise one book card. Then Python repeats the same process for every card.

### Locating the first repeated book card.

`article.product_pod` is a CSS selector:

- `article` refers to the HTML tag.
- The period means “class”.
- `product_pod` is the class name shared by every book card.
- `select_one()` returns only the first matching card.

We save that complete card in `first_book` so we can practise extracting one field at a time before looping over all books.

In [40]:
book_links = soup.select('h2 a[href*="amazon.com"]')

print("Number of book links found:", len(book_links))

Number of book links found: 95


In [41]:
first_book = book_links[0]

print(first_book)

<a href="https://www.amazon.com/gp/product/1933516666/?tag=allencheng-20" rel="nofollow" target="_blank">Life Artist</a>


`article.product_pod` means:

> Find an `article` element whose class is `product_pod`.


In [20]:
all_books = soup.select("article.product_pod")

In [21]:
all_books

[]

In [48]:
for books in all_books:
     print(books)

## 9. Extract one title

### Inside the first book card, locate the link that contains the title.

The selector `h3 a` means:

> Find an `<a>` link located inside an `<h3>` heading.

We store the matching HTML element in `title_element` and display it so we can inspect its visible text and attributes.

In [50]:
book_name = first_book

print(book_name.get_text(strip=True))

Life Artist


In [51]:
title = book_name.get_text(strip=True)

title

'Life Artist'

### The full book title is stored in the link's `title` attribute.

- `title_element["title"]` extracts that attribute's value.
- We save it in a variable called `title`.
- The final line displays the extracted title.

This is different from `get_text()`: here we are reading an HTML attribute rather than the visible words between the tags.

- `.get_text()` extracts visible text.
- `["title"]` extracts the value of an HTML attribute.


## 10. Extract the price

### Extracting the price from the first book card.

- `p.price_color` means a `<p>` element whose class is `price_color`.
- `select_one()` finds the first matching price within this book.
- `get_text(strip=True)` removes the HTML tags and unnecessary whitespace.
- The result is stored as `price_text`.

The value still includes the `£` symbol, so it is text rather than a number for now.

In [52]:
price_element = first_book.find_parent("div")

print(price_element)

<div class="col-9 col-md-12"><h2 class="display-4 mb-0 mt--2"><a href="https://www.amazon.com/gp/product/1933516666/?tag=allencheng-20" rel="nofollow" target="_blank">Life Artist</a></h2><p class="text-lg mb-0 subtitle"><a href="https://www.amazon.com/gp/product/1933516666/?tag=allencheng-20" rel="nofollow" target="_blank"></a></p><p class="justify-content-center byline pt-1"><span class="d-block d-md-inline">Creating Keepsakes</span><span class="d-none d-md-inline"> | </span><span class="sf-star-rating d-block d-md-inline"><span class="fas fa-star checked"></span><!-- --><!-- --><span class="fas fa-star checked"></span><!-- --><!-- --><span class="fas fa-star checked"></span><!-- --><!-- --><span class="fas fa-star checked"></span><!-- --><!-- --><span class="fas fa-star checked"></span></span><span> 5.00</span></p></div>


In [53]:
price_text = None

print(price_text)

None


## 11. Extract availability

### We are extracting the availability message from the first book card.

The selector `p.instock.availability` finds a paragraph that has both classes: `instock` and `availability`.

`get_text(strip=True)` returns only the visible words, such as `In stock`, and stores them in the `availability` variable.

In [54]:
availability = None

print(availability)

None


## 12. Extract the rating

### The star rating is not written as visible text. It is encoded in the element's class names.

This cell:

1. Finds the paragraph with class `star-rating`.
2. Stores the element in `rating_element`.
3. Uses `.get("class")` to retrieve its list of classes.

We expect a result such as `["star-rating", "Three"]`.

In [55]:
book_container = first_book.find_parent("div")

print(book_container.get_text(" ", strip=True))

Life Artist Creating Keepsakes | 5.00


### What we are about to do

The class list contains two items:

```python
["star-rating", "Three"]
```

Python counts list positions from zero:

- position `0` is `"star-rating"`
- position `1` is `"Three"`

This cell selects the second item and stores it as the book's rating.

In [56]:
import re

In [57]:
book_text = book_container.get_text(" ", strip=True)

rating_match = re.search(r'\b\d+\.\d+\b', book_text)

if rating_match:
    rating = float(rating_match.group())
else:
    rating = None

rating

5.0

## 13. Extract the link

### The webpage stores a relative link rather than a complete web address. `title_element.get("href")` retrieves that incomplete path.

`urljoin(url, relative_link)` combines the site's base URL with the relative path to create a complete, usable book URL.

In [58]:
relative_link = first_book.get("href")

relative_link

'https://www.amazon.com/gp/product/1933516666/?tag=allencheng-20'

In [59]:
book_url = relative_link

book_url

'https://www.amazon.com/gp/product/1933516666/?tag=allencheng-20'

## 14. Store one book as a dictionary

### Grouping the first book's extracted fields into a dictionary.

A dictionary stores information as `key: value` pairs:

- the keys will later become dataframe column names
- the values will become the cells in one row

This dictionary represents one complete observation: one book.

In [60]:
first_book_data = {
    "title": title,
    "price_text": price_text,
    "rating": rating,
    "availability": availability,
    "book_url": book_url
}

first_book_data

{'title': 'Life Artist',
 'price_text': None,
 'rating': 5.0,
 'availability': None,
 'book_url': 'https://www.amazon.com/gp/product/1933516666/?tag=allencheng-20'}

Dictionary is very close to one dataframe row:

- dictionary = row
- key = column
- value = cell

## 15. Find all book cards

### Now find **every** book card on the page.

- `select()` returns all matching elements, unlike `select_one()`, which returns only the first.
- The resulting collection is stored in `books`.
- `len(books)` counts how many cards were found.

The webpage shows 20 books, so a result of 20 is an important validation check.

In [61]:
books = soup.select('h2 a[href*="amazon.com"]')

len(books)

95

In [62]:
print("Number of books found:", len(books))

Number of books found: 95


The homepage shows 20 books, so we expect 20 matches.

In [63]:
for book in books[:10]:
    print(book.get_text(" ", strip=True))

Life Artist
Clean & Simple Designs for Scrapbooking
Clean & Simple: Scrapbooking
The Big Picture... Scrapbook Your Life and a Whole Lot More
A Designer's Eye for Scrapbooking
Creative Sketches, Volume 2
A Designer's Eye 2
The Organized and Inspired Scrapbooker
Scrapbook Page Maps
As You Grow


## 16. Loop through the books

### What we are about to do: understand the loop

`books` contains 20 separate book-card elements. Rather than copying the same title-extraction code 20 times, we use a `for` loop.

Read the first line as:

> For each individual `book` inside the collection called `books`, repeat the indented instructions.

For every card, Python:

1. gives the current card the temporary name `book`
2. searches inside that card for `h3 a`
3. extracts the `title` attribute
4. saves it temporarily as `title`
5. prints the title
6. moves to the next card and repeats

The indentation matters: both indented lines belong to the loop. When the loop finishes, all 20 titles should have been printed.

In [64]:
for book in books:
    title = book.get_text(" ", strip=True)
    print(title)

Life Artist
Clean & Simple Designs for Scrapbooking
Clean & Simple: Scrapbooking
The Big Picture... Scrapbook Your Life and a Whole Lot More
A Designer's Eye for Scrapbooking
Creative Sketches, Volume 2
A Designer's Eye 2
The Organized and Inspired Scrapbooker
Scrapbook Page Maps
As You Grow
52 More Scrapbooking Challenges (Creating Keepsakes)
Bound for Murder (A Scrapbooking Mystery, #3)
Cherish
Extraordinary Things to Cut Out and Collage
So. Many. Stickers.
Photo Decor
Her Mother's Grave
Creative Sketches for Scrapbooking
Creating Keepsakes' Encyclopedia Of Scrapbooking
Best Of Becky Higgins Sketches For Scrapbooking
Real.Life.Scrapbooking
Scrapbooking Life's Little Moments
Paper, Scissors, Death (Kiki Lowenstein Scrap-n-Craft Mystery, #1)
Keepsake Crimes (A Scrapbooking Mystery, #1)
Sharing Your Story
Creative Albums
The Bride-To-Be Book
Frill Kill (A Scrapbooking Mystery, #5)
My Creative Companion
Ultimate Guide To The Perfect Word
Le Petit Baby Book (Baby Memory Book, Baby Journal

## 17. Build a list of rows

So far we've extracted information from **one** book.

Now we want **all 20 books**.


1. Create an empty list called `rows`.
2. Visit each book card one at a time.
3. Extract the title, price, rating, availability and link.
4. Store those five pieces of information as a dictionary.
5. Add that dictionary to our growing list.

When the loop finishes, `rows` will contain one dictionary for every book.

Later, Pandas will convert that list directly into a dataframe.

In [65]:
rows = []

for rank, book in enumerate(books, start=1):

    title = book.get_text(" ", strip=True)

    book_url = book.get("href")

    container = book.find_parent("div")

    text = container.get_text(" ", strip=True)

    rating_match = re.search(r'\b\d+\.\d+\b', text)

    if rating_match:
        rating = float(rating_match.group())
    else:
        rating = None

    rows.append({
        "rank": rank,
        "title": title,
        "price_text": None,
        "rating": rating,
        "availability": None,
        "book_url": book_url
    })

len(rows)

95

### Build one row per book

This is the main scraping loop.

First, `rows = []` creates an empty list that will collect our results.

Then, for each book card, Python:

1. finds the title link
2. extracts the title
3. extracts the price
4. extracts the rating class
5. extracts availability
6. builds the full book URL
7. creates one dictionary containing those fields
8. appends that dictionary to `rows`

After the loop, `rows` should contain 20 dictionaries—one for every book. `len(rows)` checks that count.

In [34]:
# Create an empty list.
# We'll store one dictionary per book in this list.
rows = []

# Loop through every book card we found on the webpage.
# On the first iteration, 'book' is the first book.
# On the second iteration, it's the second book, and so on.
for book in books:

    # Find the <a> tag inside the <h3> tag.
    # This contains both the book title and the link to the book page.
    title_element = book.select_one("h3 a")

    # Add one dictionary to our list.
    # Each dictionary becomes one row in our final dataframe.
    rows.append({

        # The book title is stored as the 'title' attribute
        # of the <a> tag, not as visible text.
        "title": title_element["title"],

        # Find the element with class 'price_color',
        # remove extra spaces, and store the price text.
        "price_text": book.select_one(
            "p.price_color"
        ).get_text(strip=True),

        # The rating isn't written as text.
        # Instead it's stored as a class name like:
        # class="star-rating Three"
        # get("class") returns:
        # ['star-rating', 'Three']
        # [1] picks the second item ('Three').
        "rating": book.select_one(
            "p.star-rating"
        ).get("class")[1],

        # Find the availability text (e.g. "In stock")
        # and remove any extra whitespace.
        "availability": book.select_one(
            "p.instock.availability"
        ).get_text(strip=True),

        # The webpage stores a relative link such as:
        # catalogue/a-light-in-the-attic_1000/index.html
        # urljoin() combines it with the website's base URL
        # to create a complete, usable link.
        "book_url": urljoin(
            url,
            title_element.get("href")
        )
    })

# Count how many dictionaries (books) we collected.
len(rows)

20

### Inspecting the first dictionary stored in the `rows` list.

Python uses zero-based indexing, so `rows[0]` means “show the first item”. This lets us confirm that the loop stored the expected fields before we create a dataframe.

In [66]:
rows[0]

{'rank': 1,
 'title': 'Life Artist',
 'price_text': None,
 'rating': 5.0,
 'availability': None,
 'book_url': 'https://www.amazon.com/gp/product/1933516666/?tag=allencheng-20'}

## 18. Create a dataframe

This is the moment where web scraping meets Pandas.

Currently:

```
rows
```

is a **list of dictionaries**.

Pandas knows how to turn that structure into a table automatically.

Think about the mapping:

- one dictionary → one row
- dictionary keys → column names
- dictionary values → cells

After this line, everything you've already learned in Pandas works exactly the same.


### Convert the list of dictionaries into a Pandas dataframe.

Pandas interprets the structure automatically:

- each dictionary becomes one row
- each dictionary key becomes a column
- each dictionary value becomes a cell

`df.head()` then displays the first five scraped books so we can inspect the result.

In [67]:
df = pd.DataFrame(rows)

df.head()

,rank,title,price_text,rating,availability,book_url
0,1,Life Artist,None,5.00,None,https://www.amazon.com/gp/product/1933516666/?...
1,2,Clean & Simple Designs for Scrapbooking,None,4.91,None,https://www.amazon.com/gp/product/1929180616/?...
2,3,Clean & Simple: Scrapbooking,None,4.84,None,https://www.amazon.com/gp/product/1933516194/?...
3,4,The Big Picture... Scrapbook Your Life and a W...,None,4.81,None,https://www.amazon.com/gp/product/1933516798/?...
4,5,A Designer's Eye for Scrapbooking,None,4.76,None,https://www.amazon.com/gp/product/1929180683/?...


### `df.shape` reports the dataframe's dimensions as:

```text
(number of rows, number of columns)
```

We expect 20 rows because the homepage contains 20 book cards. This is another check that our scraper found the expected number of observations.

In [68]:
df.shape

(95, 6)

In [69]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 95 entries, 0 to 94
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   rank          95 non-null     int64  
 1   title         95 non-null     str    
 2   price_text    0 non-null      object 
 3   rating        95 non-null     float64
 4   availability  0 non-null      object 
 5   book_url      95 non-null     str    
dtypes: float64(1), int64(1), object(2), str(2)
memory usage: 4.6+ KB


## 19. Clean the price

### The scraped price contains a currency symbol, so Pandas currently treats it as text.

This cell creates a new numeric column:

1. select `price_text`
2. remove the `£` symbol with `.str.replace()`
3. convert the remaining text to decimal numbers using `.astype(float)`
4. save the result as `price_gbp`

We keep the original text column so the transformation remains transparent.

In [70]:
df["price_gbp"] = (
    df["price_text"]
    .str.replace("Â£", "", regex=False)
    .astype(float)
)

df[["title", "price_text", "price_gbp"]].head()

,title,price_text,price_gbp
0,Life Artist,None,NaN
1,Clean & Simple Designs for Scrapbooking,None,NaN
2,Clean & Simple: Scrapbooking,None,NaN
3,The Big Picture... Scrapbook Your Life and a W...,None,NaN
4,A Designer's Eye for Scrapbooking,None,NaN


This:

1. selects the price text
2. removes `£`
3. converts the result to a number
4. saves it as a new column


## 20. Convert ratings to numbers

### The ratings are words such as `One`, `Two` and `Five`. We want numeric values that are easier to sort, filter and summarise.

First, we create a dictionary that maps each word to a number. Then `.map(rating_map)` looks up every rating and writes the corresponding number into a new column called `rating_number`.

In [71]:
df["rating_number"] = pd.to_numeric(
    df["rating"],
    errors="coerce"
)

df[["title", "rating", "rating_number"]].head()

,title,rating,rating_number
0,Life Artist,5.00,5.00
1,Clean & Simple Designs for Scrapbooking,4.91,4.91
2,Clean & Simple: Scrapbooking,4.84,4.84
3,The Big Picture... Scrapbook Your Life and a W...,4.81,4.81
4,A Designer's Eye for Scrapbooking,4.76,4.76


In [72]:
df.head(5)

,rank,title,price_text,rating,availability,book_url,price_gbp,rating_number
0,1,Life Artist,None,5.00,None,https://www.amazon.com/gp/product/1933516666/?...,NaN,5.00
1,2,Clean & Simple Designs for Scrapbooking,None,4.91,None,https://www.amazon.com/gp/product/1929180616/?...,NaN,4.91
2,3,Clean & Simple: Scrapbooking,None,4.84,None,https://www.amazon.com/gp/product/1933516194/?...,NaN,4.84
3,4,The Big Picture... Scrapbook Your Life and a W...,None,4.81,None,https://www.amazon.com/gp/product/1933516798/?...,NaN,4.81
4,5,A Designer's Eye for Scrapbooking,None,4.76,None,https://www.amazon.com/gp/product/1929180683/?...,NaN,4.76


## 21. Analyse the scraped data

In [73]:
df.sort_values(
    "rating_number",
    ascending=False
)[[
    "title",
    "rating_number"
]].head(10)

,title,rating_number
0,Life Artist,5.00
1,Clean & Simple Designs for Scrapbooking,4.91
2,Clean & Simple: Scrapbooking,4.84
3,The Big Picture... Scrapbook Your Life and a W...,4.81
4,A Designer's Eye for Scrapbooking,4.76
5,"Creative Sketches, Volume 2",4.60
6,A Designer's Eye 2,4.58
7,The Organized and Inspired Scrapbooker,4.58
8,Scrapbook Page Maps,4.57
9,As You Grow,4.53


In [74]:
df["rating_number"].mean().round(2)

np.float64(4.23)

### Count how many books received each rating.

- `value_counts()` counts the frequency of every rating value.
- `sort_index()` arranges the result in rating order from 1 to 5 rather than by frequency.

This shows the distribution of ratings on the first page.

In [44]:
df["rating_number"].value_counts().sort_index()

rating_number
1    6
2    3
3    3
4    4
5    4
Name: count, dtype: int64

### Sorting books

1. sorts all books by the numeric price column
2. puts the most expensive books first because `ascending=False`
3. selects only the title, price and rating columns
4. displays the first ten rows

This answers a simple reporting-style question: which books on this page are most expensive?

In [75]:
df["rating_number"].value_counts().sort_index()

rating_number
4.00    3
4.01    3
4.02    2
4.03    3
4.04    3
4.05    7
4.06    6
4.07    1
4.08    3
4.09    2
4.11    4
4.13    3
4.14    4
4.15    2
4.16    2
4.17    2
4.18    4
4.19    1
4.20    1
4.22    3
4.23    2
4.25    3
4.26    3
4.28    1
4.29    1
4.31    2
4.32    1
4.33    1
4.34    1
4.35    1
4.38    1
4.41    2
4.42    1
4.43    1
4.44    2
4.45    1
4.50    1
4.52    1
4.53    1
4.57    1
4.58    2
4.60    1
4.76    1
4.81    1
4.84    1
4.91    1
5.00    1
Name: count, dtype: int64

## 22. Validate the scrape

In [77]:
print("Rows:", len(df))

print("\nMissing values:")
print(df.isna().sum())

print("\nRating range:")

print(
    df["rating_number"].min(),
    "to",
    df["rating_number"].max()
)

Rows: 95

Missing values:
rank              0
title             0
price_text       95
rating            0
availability     95
book_url          0
price_gbp        95
rating_number     0
dtype: int64

Rating range:
4.0 to 5.0


A scraper can run without an error and still collect the wrong data. Compare a few rows manually with the webpage.


## 23. Select final columns

### Create a cleaner final dataframe containing only the columns we want to keep.

`.copy()` makes an independent copy of those selected columns. This is useful because later changes to `books_df` will not accidentally modify the original `df`.

In [78]:
books_df = df[[
    "rank",
    "title",
    "price_text",
    "rating_number",
    "availability",
    "book_url"
]].copy()

books_df.head()

,rank,title,price_text,rating_number,availability,book_url
0,1,Life Artist,None,5.00,None,https://www.amazon.com/gp/product/1933516666/?...
1,2,Clean & Simple Designs for Scrapbooking,None,4.91,None,https://www.amazon.com/gp/product/1929180616/?...
2,3,Clean & Simple: Scrapbooking,None,4.84,None,https://www.amazon.com/gp/product/1933516194/?...
3,4,The Big Picture... Scrapbook Your Life and a W...,None,4.81,None,https://www.amazon.com/gp/product/1933516798/?...
4,5,A Designer's Eye for Scrapbooking,None,4.76,None,https://www.amazon.com/gp/product/1929180683/?...


## 24. Save as CSV

In [79]:
output_filename = "scrapbooking_books.csv"

books_df.to_csv(
    output_filename,
    index=False
)

print(
    f"Saved {len(books_df)} rows to {output_filename}"
)

Saved 95 rows to scrapbooking_books.csv


### Highest Rated Book


In [80]:
df.sort_values(
    "rating_number",
    ascending=False
).head(1)

,rank,title,price_text,rating,availability,book_url,price_gbp,rating_number
0,1,Life Artist,None,5.0,None,https://www.amazon.com/gp/product/1933516666/?...,NaN,5.0


### Books rated 4.5 or higher

In [81]:
len(
    df[df["rating_number"] >= 4.5]
)

12

### Median Rating

In [82]:
df["rating_number"].median()

np.float64(4.16)

## Common errors

### `No module named 'bs4'`

```bash
python3 -m pip install beautifulsoup4
```

### `AttributeError: 'NoneType' object has no attribute ...`

The selector found nothing. Inspect the element:

```python
element = book.select_one("your-selector")
print(element)
```

### Empty list

```python
books = soup.select("article.product_pod")
len(books)
```

If this returns `0`, the selector may be wrong, the page may have changed, or the content may be loaded with JavaScript.

### `KeyError: 'title'`

The selected element may not have a `title` attribute. Print it before extracting.

### Connection or timeout errors

Check the URL, your internet connection and the status code. Avoid rapid retry loops.


# Final recap

```text
URL
 ↓
requests.get()
 ↓
response.text
 ↓
BeautifulSoup()
 ↓
select repeated book cards
 ↓
extract fields
 ↓
list of dictionaries
 ↓
Pandas dataframe
 ↓
clean, validate and save
```